In [1]:
!git clone https://github.com/nlp-with-transformers/notebooks.git
%cd notebooks
from install import *
install_requirements()

fatal: destination path 'notebooks' already exists and is not an empty directory.
/content/notebooks
⏳ Installing base requirements ...
✅ Base requirements installed!
⏳ Installing Git LFS ...
✅ Git LFS installed!


In [2]:
from utils import *
setup_chapter()

No GPU was detected! This notebook can be *very* slow without a GPU 🐢
Go to Runtime > Change runtime type and select a GPU hardware accelerator.
Using transformers v4.16.2
Using datasets v1.16.1


#Making Transformers Efficient in Production

##Intent Detection as a Case Study

In [3]:
# --- Colab: Load and use a local Transformers model ---

# 1️⃣ Path to your local model folder
model_path = "/content/local_model"  # change if your folder path is different

# 2️⃣ Import libraries
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# 3️⃣ Load tokenizer and model from local folder
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

# 4️⃣ Create a pipeline
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer)

# 5️⃣ Test it with a query
query = "Hey, I'd like to rent a vehicle from Nov 1st to Nov 15th in Paris and I need a 15 passenger van"
result = pipe(query)
print(result)


OSError: Error no file named pytorch_model.bin found in directory /content/local_model but there is a file for Flax weights. Use `from_flax=True` to load this model from those weights.

In [ ]:
##Creating a Performance Benchmark

In [4]:
class PerformanceBenchmark:
  def __init__(self, pipeline, dataset, optim_type="BERT baseline"):
    self.pipeline = pipeline
    self.dataset = dataset
    self.optim_type = optim_type

  def compute_accuracy(self):
    # we'll define this later
    pass

  def time_pipeline(self):
      # We'll define this later
      pass

  def run_benchmark(self):
    metrics = {}
    metrics[self.optim_type] = self.compute_size()
    metrics[self.optim_type].update(self.time_pipeline())
    metrics[self.optim_type].update(self.compute_accuracy())
    return metrics



In [5]:
from datasets imoprt dataset

clinc = load_dataset("clinc_oos", "plus")

SyntaxError: invalid syntax (ipython-input-3700575292.py, line 1)

In [6]:
sample = clinc["test"][42]

NameError: name 'clinc' is not defined

In [7]:
intents = clinc['test'].features["intent"]
intents.int2str(sample["intent"])

NameError: name 'clinc' is not defined

In [ ]:
from datasets import load_metric

accuracy_score = load_metric("accuracy")

In [8]:
def compute_accuracy(self):
  """ This overrides the PerformanceBenchmark.compute_accuracy() method"""
  preds, labels = [], []
  for example in self.dataset:
    pred = self.pipeline(example["text"][0]["label"])
    label = example["intent"]
    preds.append(intents.str2int(pred))
    labels.append(label)
  accuracy_score = accuracy_score.compute(predictions=preds, references=labels)
  print(f"Accuracy on test set - {accuracy['accuracy']:.3f}")
  return accuracy

PerformanceBenchmark.compute_accuracy = compute_accuracy

In [9]:
list(pipe.model.state_dict().items())[42]

NameError: name 'pipe' is not defined

In [10]:
import torch
from pathlib import Path

def compute_size(self):
  """ THis overides the PerformanceBecchmark.compute_size() method"""
  state_dict = self.pipeline.model.state_dict()
  tmp_path = Path("model.pt")
  torch.save(state_dict, tmp_path)
  # Calculte the size in megabytes
  size_mb = Path(tmp_path).stat().st_size / (1024 * 1024)
  # delete the temporary file
  tmp_path.unlink()
  print(f"Model size {MB} - {size_mb:.2f}")
  return {"size_mb": size_mb}

PerformanceBenchmark.compute_size = compute_size

In [11]:
from time import perf_counter

for _ in range(3):
  start_time = perf_counter()
  _ = pipe(query)
  latency = perf_counter() - start_time
  print(f"Latency {ms} - {1000 * latency:.3f}")

NameError: name 'pipe' is not defined

In [12]:
import numpy as np

def time_pipeline(self, query="What is the pin number for my account?"):
  """ This overrides the PerformanceBenchmark.time_pipeline() method"""
  latencies = []
  # Warmup
  for _ in range(10):
    _ = self.pipeline(query)
    latency = perf_counter() - start_time
    latencies.append(latency)

  # Compute run statistics
  time_avg_ms = 1000 * np.mean(latencies)
  time_std_ms = 1000 * np.std(latencies)
  print(f"Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms: .2f}")
  return ("time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms)

PerformanceBenchmark.time_pipeline = time_pipeline

<>:15: SyntaxWarning: invalid escape sequence '\-'
<>:15: SyntaxWarning: invalid escape sequence '\-'
/tmp/ipython-input-2463261388.py:15: SyntaxWarning: invalid escape sequence '\-'
  print(f"Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms: .2f}")


SyntaxError: invalid syntax (ipython-input-2463261388.py, line 16)

In [ ]:
pb = PerformanceBenchmark(pipeline, clinc["test"])
perf_metrics = pb.run_benchmark()

##Making Models Smaller via Knowledge Distillation

##Creating a Knowledge Distillation Trainer

In [ ]:
from transformers import TrainingArguments

class DistillationTrainingArguments(TrainingArguments):
  def __init__(self, *args, alpha=0.5, temperature=2.0, **kwargs):
    super().__init__(*args, **kwargs)
    self.alpha = alpha
    self.temperature = temperature



In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

class DistillationTrainer(Trainer):
  def __init__(self, *args, teacher_model=None, **kwargs):
    super().__init__(*args, **kwargs)
    self.teacher_model = teacher_model

  def compute_loss(self, model, inputs, return_outputs=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inputs = inputs.to(device)
    outputs_stu = model(**inputs)

    # Extract cross-entropy loss asnd logits from student
    loss_ce = outputs_stu.loss
    logits_stu = outputs_stu.logits

    # Extract logits from teacher
    with torch.no_grad():
      outputs_tea = self.teacher_model(**inputs)
      logits_tea = outputs_tea.logits

    # Soften probabilities and compute the distillation loss
    loss_fct = nn.KLDivLoss(reduction="batchmean")
    loss_kd = self.args.temperature ** 2 * loss_fct(
        F.log_softmax(logits_stu / self.args.temperature, dim=1),
        F.softmax(logits_tea / self.args.temperature, dim=1))
    # Return the weghted student loss
    loss = self.args.alpha * loss_ce + (1. - self.args.alpha) * loss_kd
    return (loss, outputs_stu) if return_outputs else loss




##Choosing a Good Student Initialization


In [ ]:
from transformers import AutoTokenizer

student_ckpt = "distilbert-base-uncased"
student_tokenizer = AutoTokenizer.from_pretrained(student_ckpt)

In [ ]:
def tokenize_text(batch):
    return student_tokenizer(batch["text"], truncation=True)

clinc_enc = clinc.map(tokenize_text, batched=True, remove_columns=["text"])
clinc_enc = clinc_enc.rename_column("intent", "labels")

In [ ]:
from hugginface_hub import notebook_login

notebook_login()

In [ ]:
def compute_metrics(pred):
  predictions, labels = pred
  predictions = np.argmax(predictions, axis=1)
  return accuracy_score.compute(predictions=predictions, references=labels)

In [ ]:
batch_size = 48

finetuned_ckpt =  "distilbert-base-uncased-finetuned-clinc"
student_training_args = DistillationTrainerArguments(
    output_dir = finetuned_ckpt, evaluation_strategy = "epoch",
    num_train_epcohs = 5, learing_rate = 2e-5,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size = batch_size, alpha=1, weight_decay=0.01,
    push_to_hub=True
)

In [ ]:
student_training_args.logging_steps = len(clinc_enc['train']) // batch_size
student_training_args.disable_tqdm  = False
student_training_args.save_steps = 1e9
student_training_args.log_level = 40

In [ ]:
%env TOKENIZERS_PARALLELISM=false

In [ ]:
id2label = pipe.model.config.id2label
label2id = pipe.model.config.label2id

In [ ]:
from transformers import AutoConfig

num_labels = intents.num_classes
student_config = (AutoConfig.from_pretrained(student_ckpt, num_labels=num_labels,
                                             id2label=id2label, label2id=label2id))

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def student_init():
  return (AutoModelForSequenceClassification.from_pretrained(student_ckpt,
                                                             config=student_config).to(device))

In [ ]:
teacher_ckpt = "transformersbook/bert-base-uncased-finetuned-clinic"
teacher_model = (AutoModelForSequenceClassification.from_pretrained(teacher_ckpt, num_labels=num_labels).to(device))

In [ ]:
distilbert_trainer = DistillationTrainer(model_init=student_init,
                                         teacher_model=teacher_model, args=student_training_args,
                                         train_dataset=clinc_enc['train'], eval_dataset=clinc_enc["validation"],
                                         compute_metrics=compute_metrics, tokenizer=student_tokenizer)

distilbert_trainer.train()

In [ ]:

distilbert_trainer.push_to_hub("Training completed!")




In [ ]:
finetuned_ckpt = "transformersbook/distilbert-base-uncased-finetuned-clinc"
pipe = piepline("text-classification", model=finetuned_ckpt)

In [ ]:
optim_type = "DistilBERT"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
import pandas as pd

def plot_metrics(perf_metrics, current_optim_type):
  df = pd.DataFrame.from_dict(perf_metrics, orient="index")

  for idx in df.index:
    df_opt = df.loc[idx]
    # add a dashed circle around the current optimization type
    if idx == current_optim_type:
      plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                  alpha=0.5, s=df_opt["size_mb"], label=idx,
                  marker='\u25CC')
    else:
      plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                  s=df_opt["size_mb"], label=idx, alpha=0.5)

  legend = plt.legend(bbox_to_anchor=(1, 1))
  for handle in legend.legendHandles:
    handle.set_size([20])

  plt.ylim(80, 90)
  # Use the slowest model to define the x-axis range
  xlim = int(perf_metrics["BERT baseline"]["time_avg_ms"] + 3)
  plt.xlim(1, xlim)
  plt.ylabel("Accuracy {%}")
  plt.xlabel("Average latency {ms}")
  plt.show()
plot_metrics(perf_metrics, optim_type)

##Finding Good Hyperparameters with Optuna

In [ ]:
#hide_input
#id banana-function
#alt A banana plot
#caption Plot of the Rosenbrock function of two variables
imoprt matplotlib.pyplot as plt
import numpy as np

def f(x, y):
  return (1-x)**2+100*(y-x**2)**2

X, Y = np.meshgrid(np.linspace(-2, 2, 250), np.linspace(-1, 3, 250))
Z = f(X, Y)
_, ax = plt.subplots()
ax.plot([1], [1], 'x', mew=3, markersize=10, color="red")
ax.contourf(X, Y, Z, np.logspace(-1, 3, 30), cmap="viridis", extend="both")
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-.09, 1.7)
plt.show()

In [ ]:
def objective(trial):
    x = trial.suggest_float("x", -2, 2)
    y = trial.suggest_float("y", -2, 2)
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2



In [ ]:
import optuna

study = optuna.create_study()
study.optimize(objective, n_trials=1000)

In [ ]:
study.best_params

In [ ]:
def hp_space(trial):
  return {"num_train_epochs": trial.suggest_int("num_train_epochs", 5, 10),
          "alpha": trial.suggest_float("alpha", 0, 1) ,
          "temperature": trial.suggest_int("temperature", 2, 20)}

In [ ]:
best_run = distilbert_trainer.hyperparameter_search(
    n_trials=20, direction="maximize", hp_space=hp_space
)

In [ ]:
print(best_run)

In [ ]:
for k, v in best_run.hyperparameters.items():
   setattr(student_training_args, k, v)

# Define a new repository to store our distilled model
distilled_ckpt = "distilbert-base-uncased-distilled-clinc"
student_training_args.output_dir = distilled_ckpt

# Create a new Trainer with optimal parameters
distil_trainer = DistillationTrainer(model_init=student_init,
                                     teacher_model=teacher_model, args=student_training_args,
                                     train_dataset=clinc_enc["train"], eval_dataset=clinc_enc["validation"],
                                     compute_metrics=compute_metrics, tokenizer=student_tokenizer)

distil_trainer.train()


In [ ]:
distil_trainer.push_to_hub("Training complete")

##Benchmarking Our Distilled Model

In [ ]:
distilled_ckpt = "transformersbook/distilbert-base-uncased-distilled-clinc"
pipe = pipeline("text-classification", model=distilled_ckpt)
optim_type = "Distillation"
pb = PerformanceBenchmark(pipe, clinc['test'], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

#Making Models Faster with Quantization


In [13]:
import matplotlib.pyplot as plt

state_dict = pipe.model.state_dict()
weights = state_dict["distilbert.transformer.layer.0.attention.out_lin.weight"]
plt.hist(weights.flatten().numpy(), bins=250, range=(-0.3,0.3), edgecolor="C0")
plt.show()

NameError: name 'pipe' is not defined

In [ ]:
zero_point = 0
scale = (weights.max() - weights.min()) / (127 - (-128))


In [ ]:
(weights / scale + zero_point).clamp(-128, 127).round().char()

In [ ]:
from torch import quantize_per_tensor

dtype  = torch.qint8
qunatized_weights = quantize_per_tensor(weights, scale, zero_point, dtype)
quantized_weights.int_repr()

In [ ]:
# id weight-quantization
#alt Effect of quantization on a transformer's weights
#caption Effect of quantization on a transformer's weights

from mpl_toolkits.axes_grid1.insert_locator import zoomed_inset_axes, mark_i(nset

# Create histogram
fig, ax = plt.subplots()
ax.hist(quantized_weights.dequantize().flatten().numpy(),
        bins=250, range=(-0.3, 0.3), edgecolor="CO");


# Create a histogram

axins = zoomed_insert_axes(ax, 5, loc="upper right")
axins.hist(quantized_weights.dequantize().flatten().numpy(),
           bins=250, rang=(-0.3, 0.3));

x1, x2 , y1, y2 = 0.05, 0.1, 500, 2500
axins.set_xlim(x1, x2)
axins.set_ylim(y1, y2)
axins.axes.xaxis.set_visible(False)
axins.axes.yaxis.set_visible(False)
mark_insert(ax, axins , loc1=2, loc2=4, fc="none", ec="0.5")
plt.show()

In [ ]:
%%timeit
weights @ weights


In [ ]:
from torch.nn.qunatized import QFunctional

q_fn = QFunctional()

In [ ]:
%%timeit
q_fn.mul(quantized_weights, quantized_weights)

In [ ]:
import sys

sys.getsizeof(weights.storage()) / sys.getsizeof(quantized_weights.storage())

In [ ]:
from torch.quantization import quantize_dynamic


model_ckpt = "transformersbook/distilbert-base-uncased-distilled-clinc"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = (AutoModelForSequenceClassification
         .from_pretrained(model_ckpt).to("cpu"))

model_quantized = qunatize_dynamic(model, {nn.Linear}, dtype=torch.qint8)


##Benchmarking Our Quantized Model

In [ ]:
pipe = pipeline("text-classification", model=model_quantized,
                tokenizer=tokenizer)
optim_type = "Distillation + qunatization"
pb = PerformanceBenchmark(pipe, clinc['test'],optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

In [ ]:
import os
from psutil import cpu_count

os.environ["OMP_NUM_THREADS"] = f"{cpu_count()}"
os.environ["OMP_WAIT_POLICY"] = "ACTIVE"

In [ ]:
from transformers.convert_graph_to_onnx import convert

model_ckpt = "transformersbook/distilbert-base-uncased-distilled-clinc"
onnx_model_path = Path("onnx/model.onnx")
convert(framework="pt", model=model_ckpt, tokenizer=tokenizer,
        output=onnx_model_path, opset=12, pipeline_name="text-classification")

In [ ]:
from onnxruntime import (GraphOptimizationLevel, InferenceSession, SessionOptions)


def create_model_for_provider(model_path, provider="CPUExecutionProvider"):
  options = SessionOptions()
  options.intra_op_num_threads = 1
  options.graph_optimization_level = GraphOptimizationLevel.ORT_ENABLE_ALL
  session = InferenceSession(str(model_path), options=options, providers=[provider])
  session.disable_fallback()
  return session

onnx_model = create_model_for_provider(onnx_model_path)

In [ ]:
inputs = clinc_enc["test"][:1]
del_inputs['labels']
logits_onnx = onnx_model.run(None, inputs)[0]
logits_onnx.shape

In [ ]:
np.argmax(logits_onnx)

In [ ]:
clinc_enc["test"][0]['labels']

In [ ]:
from scipy.special import softmax

class OnnxPipeline:
  def __init__(self, model, tokenizer):
    self.model = model
    self.tokenizer = tokenizer

  def __call__(self, query):
    model_inputs = self.tokenizerr(query, return_tensors="pt")
    inputs_onnx = {k: v.cpu().detach().numpy()
                  for k , v in model_inputs.items()}
    logits = self.model.run(None, inputs_onnx)[0][0, :]
    probs = softmax(logits)
    pred_ids = np.argmax(probs).item()
    return [{"label": intents.int2str(pred_idx), "score": probs[pred_idx]}]


In [ ]:
pipe = OnnxPipeline(onnx_model, tokenizer)
pipe(query)

In [ ]:
from scipy.special import softmax

class OnnxPipeline:
  def __init__(self, model, tokenizer):
    self.model = model
    self.tokenizer = tokenizer

  def __call__(self, query):
    model_inputs = self.tokenizer(query, return_tensors="pt")
    inputs_onnx = {k: v.cpu().detach().numpy()
                  for k , v in model_inputs.items()}

    logits = self.model.run(None, inputs_onnx)[0][0, :]
    probs = softmax(logits)
    pred_idx = np.argmax(probs).item()
    return [{"label": intents.int2str(pred_idx), "score": probs[pred_idx]}]






In [ ]:
pipe = OnnxPipeline(onnx_model, tokenizer)

pipe(query)

In [ ]:
class OnnxPerformanceBenchmark(PerformanceBenchmark):
  def __init__(self, *args, model_path, **kwargs):
    super().__init__(*args, **kwargs)
    self.model_path = model_path

  def compute_size(self):
    size_mb = Path(self.model_path).stat().st_size / (1024 * 1024)
    print(f"Model size (MB - {size_mb: .2f})")
    return {"size_mb": size_mb}

In [ ]:
optim_type = "Distillation + ORT "
pb = OnnxPerfromanceBechmark(pipe, clinc["test"], optim_type,
                             model_path="onnx/model.onnx")

perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

model_input = "onnx/model.onnx"
model_output = "onnx/model.quant.onnx"
quantize_dynamic(model_input, model_output, weight_type=QuantType.QInt8)


In [ ]:
onnx_quantized_model = create_model_for_provider(model_output)
pipe = OnnxPipeline(onnx_quantized_model, tokenizer)
optim_type = "Distillaiton + ORT (quantized)"
pb = OnnxPerformanceBenchmark(pipe, clinc["test"], optim_type,
                              model_path = model_output)

perf_metrics.update(pb.run_benchmark())

In [ ]:
plot_metrics(perf_metrics, optim_type)

#Making Models Sparser with Weight Pruning

### Weight Pruning Methods


#####Magnitude pruning

In [ ]:
#id sparsity-scheduler
#alt Sparsity scheduler
#caption The cubic sparsity scheduler used for pruning
import numpy as np
import matplotlib.pyplot as plt


def _sparsity(t, t_0=0, dt=1, s_i=0, s_f=0.9,  N=100):
  return s_f + (s_i - s_f) * (1 - (t - t_0) / ( N * dt)) **3

steps = np.linspace(0, 100, 100)
values = [_sparsity(t) for t in steps]

fig, ax = plt.subplots()
ax.plot(steps, values)
ax.set_ylim(0,1)
ax.set_xlim(0,100)
ax.set_xlabel("Pruning step")
ax.set_ylabel("Sparsity")
plt.grid(linestyle="dashed")
plt.show()